In [ ]:



# =========================
# 🔗 Step 5: Merge ALL columns (FULL OUTER JOIN)
# =========================

#df = glucose_cpep_insulin.merge(glucose_cpep_insulin, on="SEQN", how="outer").merge(chol, on="SEQN", how="outer")
df = glucose_cpep_insulin.merge(
        chol,
        on="SEQN",
        how="outer",
        suffixes=("", "_CHOL")
    ).merge(
        fasting,
        on="SEQN",
        how="outer",
        suffixes=("", "_FAST")
    )

# =========================
# 🧠 Step 7: SEQN Presence Flags
# =========================
df["present_glucose_cpep_insulin"] = df["SEQN_in_glucose_cpep_insulin"].fillna(False)
df["present_chol"] = df["SEQN_in_chol"].fillna(False)
df["present_fasting"] = df["SEQN_in_fasting"].fillna(False)


display(df.head(20))
print("Unique SEQN count:", df["SEQN"].nunique())

# =========================

rename_map_glucose_cpep_insulin = {
    # ID
    "SEQN": "SEQN",
    
    # Weights
    "WTSAF2YR": "WTSAF2YR_fasting_subsample_weight_2yr",
   
    # Glucose
    "LBXGLU": "LBXGLU_plasma_glucose_mg_dl",
    "LBDGLUSI": "LBDGLUSI_plasma_glucose_mmol_l",

    # C-peptide
    "LBXCPSI": "LBXCPSI_c_peptide_si_nmol_l",

    # Insulin
    "LBXIN": "LBXIN_insulin_uu_ml",
    "LBDINSI": "LBDINSI_insulin_si_pmol_l"
}

# Safe rename (only existing columns)
df = df.rename(columns={k: v for k, v in rename_map_glucose_cpep_insulin.items() if k in df.columns})


# =========================
# 🧬 Cholesterol Rename Map (L13_C)
# =========================

rename_map_chol = {
    "SEQN": "SEQN",

    # Cholesterol (mg/dL)
    "LBXTC": "LBXTC_total_cholesterol_mg_dl",
    "LBXHDD": "LBXHDD_hdl_cholesterol_mg_dl",

    # Cholesterol (mmol/L)
    "LBDTCSI": "LBDTCSI_total_cholesterol_mmol_l",
    "LBDHDDSI": "LBDHDDSI_hdl_cholesterol_mmol_l"
}


# =========================
# 🔁 Apply AFTER merge
# =========================

df = df.rename(columns={k: v for k, v in rename_map_chol.items() if k in df.columns})



# =========================
# 🍽️ Fasting Questionnaire Rename Map (PH_C)
# =========================

rename_map_phc = {
    "SEQN": "SEQN",

    # Coffee / Tea
    "PHQ020": "PHQ020_coffee_tea_consumption_code",
    "PHACOFHR": "PHACOFHR_coffee_tea_fast_hours",
    "PHACOFMN": "PHACOFMN_coffee_tea_fast_minutes",

    # Alcohol
    "PHQ030": "PHQ030_alcohol_consumption_code",
    "PHAALCHR": "PHAALCHR_alcohol_fast_hours",
    "PHAALCMN": "PHAALCMN_alcohol_fast_minutes",

    # Gum / mints / cough drops
    "PHQ040": "PHQ040_gum_mint_cough_consumption_code",
    "PHAGUMHR": "PHAGUMHR_gum_mint_fast_hours",
    "PHAGUMMN": "PHAGUMMN_gum_mint_fast_minutes",

    # Antacids / laxatives
    "PHQ050": "PHQ050_antacid_laxative_consumption_code",
    "PHAANTHR": "PHAANTHR_antacid_fast_hours",
    "PHAANTMN": "PHAANTMN_antacid_fast_minutes",

    # Dietary supplements
    "PHQ060": "PHQ060_supplement_consumption_code",
    "PHASUPHR": "PHASUPHR_supplement_fast_hours",
    "PHASUPMN": "PHASUPMN_supplement_fast_minutes",

    # Total fasting duration
    "PHAFSTHR": "PHAFSTHR_total_fast_hours",
    "PHAFSTMN": "PHAFSTMN_total_fast_minutes",

    # Exam session
    "PHDSESN": "PHDSESN_exam_session"
}


# =========================
# 🔁 Apply AFTER merge
# =========================

df = df.rename(columns={k: v for k, v in rename_map_phc.items() if k in df.columns})


# =========================
# 👀 Step 3: Validate result
# =========================
print("📋 Final Columns After Merge + Rename:")
print(df.columns.tolist())

print("\n📊 Shape:", df.shape)


# =========================
# 💾 Step 14: Save Final Output
# =========================
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_labs_2003_2004.csv"

df.to_csv(OUTPUT_PATH, index=False)

print("\n✅ Saved to:")
print(OUTPUT_PATH)